# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors - Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and field `@id`s.

Let's inspect the structure of the dataset:


In [ ]:
# List all record sets by their @id and fields
record_sets = dataset.record_sets  # Returns a list of mlcroissant.RecordSet objects
print(f"Record sets in this dataset ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from the available record set(s) into pandas DataFrame(s) for analysis.

**Note:** All record/field/column references use their `@id` as required for compatibility and reproducibility.

In [ ]:
# Prepare to extract all record sets
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Print all record set IDs for reference
print(f"Extracting the following record sets by @id:\n{record_set_ids}\n")

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set @id: {rs_id}")

# For demonstration, select the first record set (if present)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns available in main DataFrame for {main_rs_id}:\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for further statistical analysis.


In [ ]:
# EDA operations for the main record set
# (Adapt to the dataset fields below. Fields are referenced by their @id.)

# We'll try to find a numeric column to analyze automatically
main_df = dataframes[main_rs_id]

# Show the first five rows for exploration
display(main_df.head())

# Try to detect a numeric field (by pandas type or typical column name)
numeric_field = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field = col
        break
if numeric_field is None:
    # Try common field names
    for candidate in ['age', 'age_years', 'diagnosis_interval', 'interval_months']:
        if candidate in main_df.columns:
            numeric_field = candidate
            break
if numeric_field is None:
    print("No obvious numeric field found for EDA.")
else:
    print(f"Using numeric field: {numeric_field}")

    # Choose a threshold for filtering
    threshold = main_df[numeric_field].quantile(0.5)  # Median value

    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (median):")
    display(filtered_df.head())

    # Normalize this field (z-score)
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, normalized_col]].head())

    # Try to detect a categorical/group field for grouping
    group_field = None
    for col in main_df.columns:
        if pd.api.types.is_object_dtype(main_df[col]) and (main_df[col].nunique() < len(main_df) // 3):
            group_field = col
            break
    if group_field:
        print(f"Grouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        display(grouped_df.head())
    else:
        print("No suitable group field found for demonstration.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution if we found a numeric_field
if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field], kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If we have a group field, plot boxplot
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and parse FAIR² clinical dataset metadata and tabular data using the Croissant schema and the `mlcroissant` Python library
- Enumerate available record sets, fields, and their unique `@id`s for trustworthy, reproducible workflows
- Extract and explore tabular data; filter by numerical and group characteristics
- Visualize distributions of variables, and group results by categorical attributes

This transparent, linked-data workflow encourages reuse and robust analysis of clinical oncology tabular datasets.